In [1]:
import pandas as pd

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("StudentProcessing").getOrCreate()

pdf = pd.read_csv("https://ai-ml-course.s3.us-east-1.amazonaws.com/students.csv")

df = spark.createDataFrame(pdf)

df.show()

+-----+---+-----+
| Name|Age|Score|
+-----+---+-----+
| John| 22|   80|
| Mary| 25|   90|
|Peter| 30|   70|
+-----+---+-----+



In [2]:
df.groupBy("Name").avg("Score").show()

+-----+----------+
| Name|avg(Score)|
+-----+----------+
| John|      80.0|
| Mary|      90.0|
|Peter|      70.0|
+-----+----------+



In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.sql.functions import avg
# udf - user define functions
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, DoubleType, IntegerType

In [4]:
spark = SparkSession.builder.appName("demo").getOrCreate()

In [6]:
def assign_grade(score):
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    else:
        return "F"

# implementation of the udf
grade_udf = udf(assign_grade, StringType())

In [7]:
data = [
    ("John", 85),
    ("Mary", 95),
    ("Peter", 72)
]

df = spark.createDataFrame(data, ["name", "score"])

df.show()

+-----+-----+
| name|score|
+-----+-----+
| John|   85|
| Mary|   95|
|Peter|   72|
+-----+-----+



In [8]:
df = df.withColumn("grade", grade_udf(col("score")))
df.show()

+-----+-----+-----+
| name|score|grade|
+-----+-----+-----+
| John|   85|    B|
| Mary|   95|    A|
|Peter|   72|    C|
+-----+-----+-----+



## Calculating Revenue with UDF

In [25]:
data = [
    ("Laptop", 1000, 5),
    ("Phone", 800, 10),
    ("Tablet", 500, 8)
]

columns = ["product", "price", "quantity"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+-----+--------+
|product|price|quantity|
+-------+-----+--------+
| Laptop| 1000|       5|
|  Phone|  800|      10|
| Tablet|  500|       8|
+-------+-----+--------+



In [26]:
def calculate_revenue(price, quantity):
    return float(price * quantity)


def revenue_with_tax(price, quantity):
    revenue = price * quantity
    return revenue * 1.10


def calculate_net_revenue(price, quantity, discount):
    revenue = price * quantity
    return revenue - (revenue * discount)


net_revenue_udf = udf(calculate_net_revenue,DoubleType())
revenue_udf = udf(calculate_revenue, DoubleType())
tax_udf = udf(revenue_with_tax, DoubleType())

In [27]:
df = df.withColumn("revenue",revenue_udf(df.price, df.quantity))

df.show()

+-------+-----+--------+-------+
|product|price|quantity|revenue|
+-------+-----+--------+-------+
| Laptop| 1000|       5| 5000.0|
|  Phone|  800|      10| 8000.0|
| Tablet|  500|       8| 4000.0|
+-------+-----+--------+-------+



In [28]:
df = df.withColumn("revenue_with_tax",tax_udf("price", "quantity"))

df.show()

+-------+-----+--------+-------+----------------+
|product|price|quantity|revenue|revenue_with_tax|
+-------+-----+--------+-------+----------------+
| Laptop| 1000|       5| 5000.0|          5500.0|
|  Phone|  800|      10| 8000.0|          8800.0|
| Tablet|  500|       8| 4000.0|          4400.0|
+-------+-----+--------+-------+----------------+



In [29]:
df.printSchema()

root
 |-- product: string (nullable = true)
 |-- price: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- revenue_with_tax: double (nullable = true)



In [18]:
data = [
    ("Laptop", 1000, 5, 0.10),
    ("Phone", 800, 10, 0.05),
    ("Tablet", 500, 8, 0.15)
]

df = spark.createDataFrame(data,["product", "price", "quantity", "discount"])

df = df.withColumn("net_revenue",net_revenue_udf("price","quantity","discount"))

df.show()

+-------+-----+--------+--------+-----------+
|product|price|quantity|discount|net_revenue|
+-------+-----+--------+--------+-----------+
| Laptop| 1000|       5|     0.1|     4500.0|
|  Phone|  800|      10|    0.05|     7600.0|
| Tablet|  500|       8|    0.15|     3400.0|
+-------+-----+--------+--------+-----------+



In [20]:
df.repartition(1).write.mode("overwrite").option("header", True).csv("net_revenue")

In [21]:
from pyspark.sql.functions import col

df = df.withColumn("revenue",col("price") * col("quantity"))
df.show()

+-------+-----+--------+--------+-----------+-------+
|product|price|quantity|discount|net_revenue|revenue|
+-------+-----+--------+--------+-----------+-------+
| Laptop| 1000|       5|     0.1|     4500.0|   5000|
|  Phone|  800|      10|    0.05|     7600.0|   8000|
| Tablet|  500|       8|    0.15|     3400.0|   4000|
+-------+-----+--------+--------+-----------+-------+



In [22]:
df = df.withColumn("net_revenue",(col("price") * col("quantity")) * (1 - col("discount")))

df.show()

+-------+-----+--------+--------+-----------+-------+
|product|price|quantity|discount|net_revenue|revenue|
+-------+-----+--------+--------+-----------+-------+
| Laptop| 1000|       5|     0.1|     4500.0|   5000|
|  Phone|  800|      10|    0.05|     7600.0|   8000|
| Tablet|  500|       8|    0.15|     3400.0|   4000|
+-------+-----+--------+--------+-----------+-------+



In [32]:
@udf(returnType=IntegerType())
def calc(a, b):
    return a + 10 * b

In [33]:
def calc(a, b):
    return a + 10 * b

calc_udf = udf(calc, IntegerType())

In [34]:
spark.range(2).select(calc(b=col("id") * 10, a=col("id"))).show()

+-----------------------+
|(id + ((id * 10) * 10))|
+-----------------------+
|                      0|
|                    101|
+-----------------------+



In [35]:
from pyspark.sql.functions import pandas_udf
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.sql.functions import avg
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql.types import DoubleType


spark = SparkSession.builder.appName("demo").getOrCreate()


data = [
    ("John", "Doe"),
    ("Mary", "Smith")
]

df = spark.createDataFrame(data, ["first_name", "last_name"])

@pandas_udf("string")
def upper_pandas(col: pd.Series) -> pd.Series:
    return col.str.upper()

# Example usage:
df.withColumn("upper_name", upper_pandas("first_name")).show()

+----------+---------+----------+
|first_name|last_name|upper_name|
+----------+---------+----------+
|      John|      Doe|      JOHN|
|      Mary|    Smith|      MARY|
+----------+---------+----------+

